# Notebook 05 — Preparación de Datos y Modelado ML

**Proyecto:** Sistema de Predicción y Clasificación de la Desnutrición en niños menores de cinco años  
**Fase CRISP-DM:** 3 (Preparación) + 4 (Modelado)  
**Dataset:** `dataset_ml.csv` (generado por notebook 03)  
**Variable objetivo:** `clas_peso`

---
## Contenido
### Parte 1 — Preparación de datos
1. Carga e inspección
2. Ingeniería de features — selección y descarte justificado
3. Codificación de variables categóricas
4. Imputación de nulos residuales
5. Escalado — RobustScaler selectivo
6. División train/test estratificada
7. Balanceo de clases — SMOTE

### Parte 2 — Modelado
8. Entrenamiento de modelos — **Modelo A (con IMC)**
9. Comparativa de métricas — Modelo A
10. Matriz de confusión — Modelo A
11. Feature importance — Modelo A
12. **Modelo B — Sin IMC (contexto rural / atención primaria)**
13. Comparativa final: Modelo A vs Modelo B
14. Conclusiones y modelo seleccionado

---
# PARTE 1 — Preparación de datos

## 1. Carga e inspección

In [ ]:
import sys
!{sys.executable} -m pip install imbalanced-learn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, f1_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score)
from imblearn.over_sampling import SMOTE

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})

CLASES     = ['Desnut. severa', 'Desnut. moderada', 'Normal bajo',
              'Normal', 'Sobrepeso', 'Obesidad']
COLORES    = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60', '#2980b9', '#8e44ad']

df_raw = pd.read_csv('../data/processed/dataset_ml.csv')
print(f'Shape original : {df_raw.shape}')
print(f'Columnas       : {list(df_raw.columns)}')
print(f'\nclas_peso distribución:')
print(df_raw['clas_peso'].value_counts().sort_index())

## 2. Ingeniería de features — selección y descarte justificado

Se parte de las 53 columnas del `dataset_ml.csv` y se reduce a un set limpio de features,
eliminando columnas por razones metodológicas documentadas.

> **Nota sobre el IMC:** Se desarrollan **dos versiones del modelo**:
> - **Modelo A (con IMC):** para clínicas y hospitales con báscula y tallímetro.
>   El IMC es una medida clínica estándar calculable en el momento de la consulta (peso/talla²).
>   A diferencia de `zscore_pt`, no requiere tablas de referencia OMS ni define directamente `clas_peso`.
> - **Modelo B (sin IMC):** para promotores de salud en zonas rurales sin equipo completo.
>   Solo requiere cinta métrica para el perímetro braquial. Más útil para detección temprana en campo.

In [ ]:
# ── Eliminar clas_peso nulos (9 registros) ────────────────────────────────
df = df_raw.dropna(subset=['clas_peso']).copy()
df['clas_peso'] = df['clas_peso'].astype(int)

# ── Crear edad_meses (más informativa que edad_ + uni_med_ por separado) ──
df['edad_meses'] = df.apply(
    lambda r: r['edad_'] * 12 if r['uni_med_'] == 1 else r['edad_'], axis=1
)

# ── Columnas a eliminar y justificación ───────────────────────────────────
DESCARTAR = {
    'talla_act'   : 'Multicolinealidad con peso_act (r=0.95) — aporta información redundante',
    'talla_nac'   : '61% nulos — no es imputable de forma fiable',
    'anio'        : 'Variable temporal administrativa, no feature del paciente',
    'edad_'       : 'Reemplazada por edad_meses (más precisa)',
    'uni_med_'    : 'Solo necesaria para calcular edad_meses, ya creada',
    'nacionali_'  : 'Redundante con cod_pais_o y per_etn_',
    'cod_pais_o'  : 'Redundante con per_etn_ y area_',
    'cod_pais_r'  : 'Redundante',
    'cod_mun_o'   : 'Alta cardinalidad (36 municipios) — usar departamento',
    'cod_mun_r'   : 'Alta cardinalidad (39 municipios)',
    'nmun_resi'   : 'Alta cardinalidad (44 valores) — texto libre',
    'cod_dpto_r'  : 'Redundante con cod_dpto_o',
    'ndep_resi'   : 'Texto — reemplazado por cod_dpto_o codificado',
    'tip_ss_'     : '97% del mismo valor (S=subsidiado) — sin variabilidad',
    'gp_desplaz'  : 'Solo 9 casos positivos — sin variabilidad estadística',
    'gp_migrant'  : 'Solo 10 casos positivos',
    'gp_indigen'  : 'Solo 1 caso positivo',
    'gp_vic_vio'  : 'Solo 1 caso positivo',
    'gp_mad_com'  : 'Todos el mismo valor (2) — sin variabilidad',
    'gp_gestan'   : 'Todos el mismo valor (2) — sin variabilidad',
    'tipo_manej'  : '178 nulos + redundante con ruta_atenc',
    'con_fin_'    : 'Variable de resultado — no disponible al momento de predecir',
    'pac_hos_'    : 'Variable de resultado — no disponible al momento de predecir',
    'fuente_'     : 'Variable administrativa',
    'tip_cas_'    : 'Variable administrativa',
    # NOTA: zscore_pt y zscore_te ya fueron excluidos en el notebook 03 (data leakage puro)
    # NOTA: imc se mantiene en Modelo A — es calculable en consulta (peso/talla²)
    #        y se excluye en Modelo B para simular contexto rural sin equipos completos
}

print('Columnas descartadas y justificación:')
for col, razon in DESCARTAR.items():
    print(f'  {col:20s}: {razon}')

# ── Features Modelo A — CON IMC ───────────────────────────────────────────
FEATURES_A = [
    'edad_meses', 'per_etn_', 'estrato_',
    'area_', 'cod_dpto_o',
    'niv_educat', 'menores', 'gp_pobicbf',
    'peso_nac', 'edad_ges',
    'peso_act', 'per_braqui', 'imc',        # ← imc incluido
    't_lechem', 'e_complem',
    'crec_dllo', 'esq_vac', 'carne_vac',
    'edema', 'delgadez', 'palidez',
    'piel_rese', 'hiperpigm', 'cambios_cabello',
    'ruta_atenc',
]

# ── Features Modelo B — SIN IMC ───────────────────────────────────────────
FEATURES_B = [f for f in FEATURES_A if f != 'imc']

faltantes = [f for f in FEATURES_A if f not in df.columns]
print(f'\nFeatures faltantes: {faltantes if faltantes else "Ninguno ✅"}')
print(f'Modelo A (con IMC) : {len(FEATURES_A)} features')
print(f'Modelo B (sin IMC) : {len(FEATURES_B)} features')

## 3. Codificación de variables categóricas

- **`cod_dpto_o`** — viene como string (`'20'`, `'44'`, `'D0'`). Se aplica **Label Encoding**
  porque es ordinal implícito y tiene baja cardinalidad (9 valores únicos).
- El resto de variables categóricas ya son numéricas en el dataset.

In [ ]:
# Preparar df_model base (aplica a ambos modelos)
df_model = df[FEATURES_A + ['clas_peso']].copy()

# Label Encoding para cod_dpto_o (string → entero)
le_dpto = LabelEncoder()
df_model['cod_dpto_o'] = le_dpto.fit_transform(df_model['cod_dpto_o'].astype(str))

print('Codificación cod_dpto_o:')
for codigo, etiqueta in zip(le_dpto.classes_, le_dpto.transform(le_dpto.classes_)):
    print(f'  {codigo} → {etiqueta}')

print(f'\nDtypes resultantes:')
print(df_model.dtypes.to_string())

## 4. Imputación de nulos residuales

Solo 3 features tienen nulos residuales tras el ETL:
- `niv_educat` (33 nulos) → mediana
- `gp_pobicbf` (2 nulos) → valor por defecto 2 (No)
- `per_braqui` (227 nulos, ~9%) → mediana por `clas_peso` (imputación estratificada)

In [ ]:
print('Nulos antes de imputación:')
nulos = df_model.isnull().sum()
print(nulos[nulos > 0].to_string())

# niv_educat: mediana global
med_edu = df_model['niv_educat'].median()
df_model['niv_educat'] = df_model['niv_educat'].fillna(med_edu)
print(f'\nniv_educat: imputado con mediana = {med_edu}')

# gp_pobicbf: 2 = No (valor por defecto)
df_model['gp_pobicbf'] = df_model['gp_pobicbf'].fillna(2)
print('gp_pobicbf: 2 nulos → 2 (No)')

# menores: mediana global
df_model['menores'] = df_model['menores'].fillna(df_model['menores'].median())

# per_braqui: mediana estratificada por clas_peso
df_model['per_braqui'] = df_model.groupby('clas_peso')['per_braqui'].transform(
    lambda x: x.fillna(x.median())
)
df_model['per_braqui'] = df_model['per_braqui'].fillna(df_model['per_braqui'].median())
print('per_braqui: imputado con mediana estratificada por clas_peso')

print(f'\nNulos después de imputación: {df_model.isnull().sum().sum()} ✅')

## 5. Escalado — RobustScaler selectivo

Se aplica **RobustScaler** solo a las variables que resultaron NO normales en las pruebas
de normalidad del notebook 04 (sección 12). Las variables ya normales no se tocan.

**¿Por qué RobustScaler y no StandardScaler?**  
Porque usa la mediana y el rango intercuartílico en lugar de la media y desviación estándar,
lo que lo hace resistente a los outliers que vimos en los boxplots del notebook 04.

In [ ]:
# Variables NO normales → escalar (aplica a ambos modelos)
COLS_ESCALAR = ['peso_act', 'per_braqui', 't_lechem', 'menores',
                'peso_nac', 'edad_meses', 'imc', 'edad_ges', 'e_complem']

# ── Modelo A — con IMC ────────────────────────────────────────────────────
X_A = df_model[FEATURES_A].copy()
y   = df_model['clas_peso'].copy()

scaler_A = RobustScaler()
cols_esc_A = [c for c in COLS_ESCALAR if c in FEATURES_A]
X_A[cols_esc_A] = scaler_A.fit_transform(X_A[cols_esc_A])

# ── Modelo B — sin IMC ────────────────────────────────────────────────────
X_B = df_model[FEATURES_B].copy()

scaler_B = RobustScaler()
cols_esc_B = [c for c in COLS_ESCALAR if c in FEATURES_B]
X_B[cols_esc_B] = scaler_B.fit_transform(X_B[cols_esc_B])

print('Escalado aplicado:')
print(f'  Modelo A ({len(FEATURES_A)} features): {cols_esc_A}')
print(f'  Modelo B ({len(FEATURES_B)} features): {cols_esc_B}')
print(f'\nShape X_A: {X_A.shape}')
print(f'Shape X_B: {X_B.shape}')

## 6. División train/test estratificada

Se usa **stratify=y** para garantizar que la proporción de cada clase de desnutrición
sea la misma en train y test. Sin esto, las clases minoritarias (Sobrepeso, Obesidad)
podrían quedar subrepresentadas en el conjunto de prueba.

In [ ]:
# Split estratificado — Modelo A
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X_A, y, test_size=0.2, random_state=42, stratify=y
)

# Split estratificado — Modelo B (mismo índice para comparación justa)
X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
    X_B, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train_A.shape[0]:,} registros (80%)')
print(f'Test  : {X_test_A.shape[0]:,} registros (20%)')
print(f'\nDistribución de clases en train:')
print(y_train_A.value_counts().sort_index().rename('Train'))
print(f'\nDistribución de clases en test:')
print(y_test_A.value_counts().sort_index().rename('Test'))

## 7. Balanceo de clases — SMOTE

El dataset tiene un desbalance severo: >85% de los casos son desnutrición.
Las clases Sobrepeso (24 casos) y Obesidad (6 casos) son muy minoritarias.

**SMOTE** (Synthetic Minority Over-sampling Technique) genera ejemplos sintéticos
de las clases minoritarias interpolando entre casos reales existentes.

**Importante:** SMOTE se aplica **solo al conjunto de entrenamiento**, nunca al test.
Aplicarlo al test contaminaría la evaluación.

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=3)

X_train_A_sm, y_train_A_sm = smote.fit_resample(X_train_A, y_train_A)
X_train_B_sm, y_train_B_sm = smote.fit_resample(X_train_B, y_train_B)

print('Distribución ANTES de SMOTE (train):')
print(y_train_A.value_counts().sort_index().to_string())
print(f'\nDistribución DESPUÉS de SMOTE:')
print(pd.Series(y_train_A_sm).value_counts().sort_index().to_string())
print(f'\nTamaño train original : {len(y_train_A):,}')
print(f'Tamaño train con SMOTE: {len(y_train_A_sm):,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
y_train_A.value_counts().sort_index().plot(kind='bar', ax=axes[0], color=COLORES, edgecolor='white')
axes[0].set_title('Antes de SMOTE', fontsize=12)
axes[0].set_xticklabels(CLASES, rotation=20, ha='right')
axes[0].set_ylabel('Casos')

pd.Series(y_train_A_sm).value_counts().sort_index().plot(kind='bar', ax=axes[1], color=COLORES, edgecolor='white')
axes[1].set_title('Después de SMOTE', fontsize=12)
axes[1].set_xticklabels(CLASES, rotation=20, ha='right')
axes[1].set_ylabel('Casos')
plt.suptitle('Distribución de clases en train — antes y después de SMOTE', fontsize=13)
plt.tight_layout()
plt.show()

---
# PARTE 2 — Modelado

## 8. Entrenamiento de modelos — Modelo A (con IMC)

Se entrenan 4 modelos con los **26 features incluyendo IMC**:

| Modelo | Tipo | Razón de inclusión |
|---|---|---|
| Logistic Regression | Lineal | Baseline — modelo simple de referencia |
| Random Forest | Ensamble de árboles | Robusto, maneja bien variables mixtas |
| Gradient Boosting | Ensamble boosting | Generalmente el mejor en datos tabulares |
| SVM | Kernel | Efectivo en espacios de alta dimensión |

In [ ]:
MODELOS_DEF = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42, C=1.0
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        max_depth=None, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1,
        max_depth=4, random_state=42
    ),
    'SVM': SVC(
        kernel='rbf', class_weight='balanced',
        C=10, gamma='scale', probability=True, random_state=42
    ),
}

modelos_A = {}
for nombre, modelo in MODELOS_DEF.items():
    print(f'Entrenando {nombre} (Modelo A)...', end=' ')
    modelo.fit(X_train_A_sm, y_train_A_sm)
    modelos_A[nombre] = modelo
    print('✅')

## 9. Comparativa de métricas — Modelo A (con IMC)

- **F1-weighted**: métrica principal — pondera por frecuencia de clase
- **F1-macro**: sin ponderar — penaliza errores en clases minoritarias
- **Recall-D.severa**: % de casos severos correctamente detectados — crítico en salud pública

In [ ]:
def evaluar_modelos(modelos, X_test, y_test, titulo):
    resultados = []
    predicciones = {}
    for nombre, modelo in modelos.items():
        y_pred = modelo.predict(X_test)
        predicciones[nombre] = y_pred
        acc   = (y_pred == y_test).mean()
        f1_w  = f1_score(y_test, y_pred, average='weighted')
        f1_m  = f1_score(y_test, y_pred, average='macro')
        f1_c1 = f1_score(y_test, y_pred, labels=[1], average='macro')
        rec_c1 = (y_pred[y_test == 1] == 1).mean() if (y_test == 1).sum() > 0 else 0
        resultados.append({
            'Modelo'          : nombre,
            'Accuracy'        : round(acc,   4),
            'F1-weighted'     : round(f1_w,  4),
            'F1-macro'        : round(f1_m,  4),
            'F1-Desnut.severa': round(f1_c1, 4),
            'Recall-D.severa' : round(rec_c1, 4),
        })
    df_res = pd.DataFrame(resultados).sort_values('F1-weighted', ascending=False)

    print(f'\n{titulo}')
    print(df_res.to_string(index=False))

    fig, ax = plt.subplots(figsize=(11, 5))
    metricas = ['Accuracy','F1-weighted','F1-macro','F1-Desnut.severa','Recall-D.severa']
    x = np.arange(len(metricas))
    width = 0.2
    colores_mod = ['#185fa5','#27ae60','#e67e22','#8e44ad']
    for i, (_, row) in enumerate(df_res.iterrows()):
        vals = [row[m] for m in metricas]
        ax.bar(x + i*width, vals, width, label=row['Modelo'],
               color=colores_mod[i], edgecolor='white', alpha=0.85)
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(metricas, rotation=15, ha='right')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.1)
    ax.set_title(titulo, fontsize=13, pad=12)
    ax.legend(loc='lower right', framealpha=0.5)
    plt.tight_layout()
    plt.show()
    return df_res, predicciones

df_res_A, preds_A = evaluar_modelos(modelos_A, X_test_A, y_test_A,
                                     'Modelo A (con IMC) — comparativa de métricas')

In [ ]:
for nombre, y_pred in preds_A.items():
    print(f'\n{"="*55}')
    print(f'  {nombre} — Modelo A (con IMC)')
    print(f'{"="*55}')
    print(classification_report(y_test_A, y_pred, target_names=CLASES, zero_division=0))

## 10. Matriz de confusión — Modelo A (con IMC)

- Diagonal principal → predicciones correctas
- Fuera de la diagonal → errores
- El error más crítico: confundir **Desnutrición severa** con moderada

In [ ]:
mejor_A = df_res_A.iloc[0]['Modelo']
y_pred_mejor_A = preds_A[mejor_A]

print(f'Mejor modelo A: {mejor_A} | F1-weighted: {df_res_A.iloc[0]["F1-weighted"]:.4f}')

cm_A = confusion_matrix(y_test_A, y_pred_mejor_A)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm_A, display_labels=CLASES).plot(
    ax=ax, cmap='Blues', colorbar=True, xticks_rotation=25)
ax.set_title(f'Matriz de confusión — {mejor_A} — Modelo A (con IMC)', fontsize=12, pad=12)
plt.tight_layout()
plt.show()

total_severos = (y_test_A == 1).sum()
detectados_A  = cm_A[0, 0]
print(f'\nDesnutrición severa — Modelo A:')
print(f'  Casos reales     : {total_severos}')
print(f'  Detectados       : {detectados_A} ({detectados_A/total_severos*100:.1f}%)')
print(f'  No detectados FN : {total_severos - detectados_A} ({(total_severos-detectados_A)/total_severos*100:.1f}%)')

## 11. Feature importance — Modelo A (con IMC)

In [ ]:
modelo_mejor_A = modelos_A[mejor_A]

if hasattr(modelo_mejor_A, 'feature_importances_'):
    imp_A = pd.Series(modelo_mejor_A.feature_importances_, index=FEATURES_A)
elif hasattr(modelo_mejor_A, 'coef_'):
    imp_A = pd.Series(np.abs(modelo_mejor_A.coef_).mean(axis=0), index=FEATURES_A)
else:
    rf_A = modelos_A.get('Random Forest')
    imp_A = pd.Series(rf_A.feature_importances_, index=FEATURES_A)

imp_A = imp_A.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colores_fi = ['#c0392b' if i < 5 else '#185fa5' if i < 15 else '#95a5a6'
              for i in range(len(imp_A))]
imp_A[::-1].plot(kind='barh', ax=ax, color=colores_fi[::-1], edgecolor='white')
ax.set_title(f'Feature Importance — {mejor_A} — Modelo A (con IMC)\n(rojo = top 5)',
             fontsize=12, pad=12)
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()
print(imp_A.round(4).to_string())

In [ ]:
print('Comparativa ANOVA (notebook 04) vs Feature Importance Modelo A:')
print('='*65)
anova_f = {
    'per_braqui': 87.87, 'peso_act': 55.36, 'imc': '⚠️ (parcial)',
    't_lechem': 11.37, 'niv_educat': 9.83, 'edad_meses': 5.85,
    'area_': 4.77, 'peso_nac': 4.50, 'menores': 1.83, 'estrato_': 1.75
}
for var, f_val in anova_f.items():
    if var in imp_A.index:
        imp = imp_A[var]
        print(f'{var:20s}  ANOVA-F: {str(f_val):12s}  Importancia: {imp:.4f}')

---
## 12. Modelo B — Sin IMC (contexto rural / atención primaria)

**¿Por qué un modelo sin IMC?**

El IMC requiere báscula y tallímetro calibrados. En zonas rurales dispersas del Cesar
y La Guajira, los promotores de salud a menudo solo disponen de una cinta métrica.

Este modelo usa las mismas variables excepto `imc`, y evalúa si sigue siendo útil
para la detección temprana en campo con recursos limitados.

In [ ]:
import copy

modelos_B = {}
for nombre, modelo_orig in MODELOS_DEF.items():
    modelo_b = copy.deepcopy(modelo_orig)
    print(f'Entrenando {nombre} (Modelo B)...', end=' ')
    modelo_b.fit(X_train_B_sm, y_train_B_sm)
    modelos_B[nombre] = modelo_b
    print('✅')

In [ ]:
df_res_B, preds_B = evaluar_modelos(modelos_B, X_test_B, y_test_B,
                                     'Modelo B (sin IMC) — comparativa de métricas')

In [ ]:
# Reporte detallado modelo B
for nombre, y_pred in preds_B.items():
    print(f'\n{"="*55}')
    print(f'  {nombre} — Modelo B (sin IMC)')
    print(f'{"="*55}')
    print(classification_report(y_test_B, y_pred, target_names=CLASES, zero_division=0))

In [ ]:
# Matriz de confusión mejor modelo B
mejor_B = df_res_B.iloc[0]['Modelo']
y_pred_mejor_B = preds_B[mejor_B]

print(f'Mejor modelo B: {mejor_B} | F1-weighted: {df_res_B.iloc[0]["F1-weighted"]:.4f}')

cm_B = confusion_matrix(y_test_B, y_pred_mejor_B)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm_B, display_labels=CLASES).plot(
    ax=ax, cmap='Greens', colorbar=True, xticks_rotation=25)
ax.set_title(f'Matriz de confusión — {mejor_B} — Modelo B (sin IMC)', fontsize=12, pad=12)
plt.tight_layout()
plt.show()

detectados_B = cm_B[0, 0]
print(f'\nDesnutrición severa — Modelo B:')
print(f'  Detectados : {detectados_B} ({detectados_B/total_severos*100:.1f}%)')
print(f'  No detectados FN: {total_severos - detectados_B} ({(total_severos-detectados_B)/total_severos*100:.1f}%)')

In [ ]:
# Feature importance Modelo B
modelo_mejor_B = modelos_B[mejor_B]

if hasattr(modelo_mejor_B, 'feature_importances_'):
    imp_B = pd.Series(modelo_mejor_B.feature_importances_, index=FEATURES_B)
elif hasattr(modelo_mejor_B, 'coef_'):
    imp_B = pd.Series(np.abs(modelo_mejor_B.coef_).mean(axis=0), index=FEATURES_B)
else:
    rf_B = modelos_B.get('Random Forest')
    imp_B = pd.Series(rf_B.feature_importances_, index=FEATURES_B)

imp_B = imp_B.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colores_fi_B = ['#c0392b' if i < 5 else '#27ae60' if i < 15 else '#95a5a6'
                for i in range(len(imp_B))]
imp_B[::-1].plot(kind='barh', ax=ax, color=colores_fi_B[::-1], edgecolor='white')
ax.set_title(f'Feature Importance — {mejor_B} — Modelo B (sin IMC)\n'
             f'(rojo = top 5 | sin IMC el protagonismo pasa a per_braqui)',
             fontsize=12, pad=12)
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()
print(imp_B.round(4).to_string())

## 13. Comparativa final: Modelo A vs Modelo B

In [ ]:
# Comparativa directa A vs B con el mismo algoritmo
print('COMPARATIVA FINAL — Modelo A (con IMC) vs Modelo B (sin IMC)')
print('='*65)
print(f'{"Métrica":<22} {"Modelo A (con IMC)":>20} {"Modelo B (sin IMC)":>20} {"Diferencia":>12}')
print('-'*65)

metricas_comp = ['Accuracy','F1-weighted','F1-macro','F1-Desnut.severa','Recall-D.severa']
fila_A = df_res_A[df_res_A['Modelo'] == mejor_A].iloc[0]
fila_B = df_res_B[df_res_B['Modelo'] == mejor_B].iloc[0]

for m in metricas_comp:
    val_A = fila_A[m]
    val_B = fila_B[m]
    diff  = val_A - val_B
    signo = '+' if diff >= 0 else ''
    print(f'{m:<22} {val_A:>20.4f} {val_B:>20.4f} {signo+str(round(diff,4)):>12}')

print()
print(f'Mejor algoritmo Modelo A: {mejor_A}')
print(f'Mejor algoritmo Modelo B: {mejor_B}')

In [ ]:
# Gráfico comparativo A vs B
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metricas_comp))
width = 0.35

vals_A = [fila_A[m] for m in metricas_comp]
vals_B = [fila_B[m] for m in metricas_comp]

bars_A = ax.bar(x - width/2, vals_A, width, label=f'Modelo A — {mejor_A} (con IMC)',
                color='#185fa5', edgecolor='white', alpha=0.85)
bars_B = ax.bar(x + width/2, vals_B, width, label=f'Modelo B — {mejor_B} (sin IMC)',
                color='#27ae60', edgecolor='white', alpha=0.85)

for bar in bars_A:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)
for bar in bars_B:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metricas_comp, rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_title('Comparativa Modelo A (con IMC) vs Modelo B (sin IMC)\n'
             'azul = con IMC | verde = sin IMC', fontsize=13, pad=12)
ax.legend(framealpha=0.5)
plt.tight_layout()
plt.show()

print('\nInterpretación:')
print('  La diferencia entre ambos modelos indica el "costo" de no tener IMC.')
print('  Si la diferencia es pequeña (<0.05), el Modelo B es viable en campo.')
print('  Si es grande (>0.10), se recomienda priorizar el Modelo A en clínicas.')

## 14. Conclusiones y modelo seleccionado

### Dos modelos para dos contextos

| | Modelo A (con IMC) | Modelo B (sin IMC) |
|---|---|---|
| **Contexto** | Clínicas y hospitales | Promotores rurales / atención primaria |
| **Equipos necesarios** | Báscula + tallímetro | Solo cinta métrica |
| **Predictor principal** | IMC → per_braqui | per_braqui → peso_act |
| **Recomendado para** | Diagnóstico clínico formal | Detección temprana en campo |

### Hallazgos del modelado

| Aspecto | Resultado |
|---|---|
| Mejor algoritmo general | Gradient Boosting |
| Mejor predictor Modelo A | `imc` + `per_braqui` |
| Mejor predictor Modelo B | `per_braqui` + `peso_act` |
| Variables maternas con aporte | `t_lechem`, `niv_educat` |
| Variables sin aporte (ambos) | `menores`, `estrato_`, `crec_dllo` |
| Desbalance manejado con | SMOTE (solo en train) |
| Escalado aplicado | RobustScaler selectivo |

### Sobre el data leakage
- `zscore_pt` y `zscore_te` → excluidos del `dataset_ml.csv` en notebook 03 (leakage puro — definen directamente `clas_peso` según Res. 2465/2016)
- `imc` → incluido en Modelo A (medida clínica calculable), excluido en Modelo B (requiere equipos)

### Próximos pasos
- **Notebook 06:** Predicción temporal con `serie_temporal_mensual.csv`
- **Notebook 07:** Comparativa territorial Cesar vs La Guajira vs Magdalena
- **Opcional:** GridSearchCV para ajuste de hiperparámetros del mejor modelo